In [1]:
import os
import ROOT
import numpy as np
import pandas as pd
from sklearn.utils import shuffle

Welcome to JupyROOT 6.28/04


In [2]:
#### global variables
WORKDIR = "/home/choij/workspace/SNU-project/4.SNU-CMS/Tutorials/MachineLearning"
SPLIT = "train"     # train / valid / test

In [3]:
# configurations for input features
configs = {
    "j1_energy": (500, 0., 500.),
    "j1_px": (400, -200., 200.),
    "j1_py": (400, -200., 200.),
    "j1_pz": (300, -150., 150.),
    "j1_chEmEF": (100, 0., 1.),
    "j1_chHEF": (100, 0., 1.),
    "j1_neEmEF": (100, 0., 1.),
    "j1_neHEF": (100, 0., 1.),
    "j1_muEF": (100, 0., 1.),
    "j1_btagDeepFlavB": (100, 0., 1.),
    "j1_btagDeepFlavQG": (100, 0., 1.),
    "j2_energy": (300, 0., 300.),
    "j2_px": (300, -150., 150.),
    "j2_py": (300, -150., 150.),
    "j2_pz": (300, -150., 150.),
    "j2_chEmEF": (100, 0., 1.),
    "j2_chHEF": (100, 0., 1.),
    "j2_neEmEF": (100, 0., 1.),
    "j2_neHEF": (100, 0., 1.),
    "j2_muEF": (100, 0., 1.),
    "j2_btagDeepFlavB": (100, 0., 1.),
    "j2_btagDeepFlavQG": (100, 0., 1.),
    "j3_energy": (300, 0., 300.),
    "j3_px": (300, -150., 150.),
    "j3_py": (300, -150., 150.),
    "j3_pz": (300, -150., 150.),
    "j3_chEmEF": (100, 0., 1.),
    "j3_chHEF": (100, 0., 1.),
    "j3_neEmEF": (100, 0., 1.),
    "j3_neHEF": (100, 0., 1.),
    "j3_muEF": (100, 0., 1.),
    "j3_btagDeepFlavB": (100, 0., 1.),
    "j3_btagDeepFlavQG": (100, 0., 1.),
    "j4_energy": (300, 0., 300.),
    "j4_px": (300, -150., 150.),
    "j4_py": (300, -150., 150.),
    "j4_pz": (300, -150., 150.),
    "j4_chEmEF": (100, 0., 1.),
    "j4_chHEF": (100, 0., 1.),
    "j4_neEmEF": (100, 0., 1.),
    "j4_neHEF": (100, 0., 1.),
    "j4_muEF": (100, 0., 1.),
    "j4_btagDeepFlavB": (100, 0., 1.),
    "j4_btagDeepFlavQG": (100, 0., 1.),
    "avg_deltaR": (100, 0., 10.),
    "Nj": (20, 0., 20.),
    "HT": (100, 0., 1000.)
}

In [4]:
# book histograms
sigHists = {}
bkgHists = {}
for feature, (nbins, xL, xR) in configs.items():
    sigHist = ROOT.TH1D(f"sig_{feature}", "", nbins, xL, xR); sigHist.SetDirectory(0)
    bkgHist = ROOT.TH1D(f"bkg_{feature}", "", nbins, xL, xR); bkgHist.SetDirectory(0)
    sigHists[feature] = sigHist
    bkgHists[feature] = bkgHist

In [5]:
# Load dataset
sigCSV = pd.read_csv(f"{WORKDIR}/DATA/TT/sample.csv", index_col=0); sigCSV["label"] = 1
bkgCSV = pd.read_csv(f"{WORKDIR}/DATA/QCD/sample.csv", index_col=0); bkgCSV["label"] = 0

sample = shuffle(pd.concat([sigCSV, bkgCSV], ignore_index=True), random_state=42)
trainset = sample[:int(len(sample)*0.6)].copy()
validset = sample[int(len(sample)*0.6):int(len(sample)*0.7)].copy()
testset = sample[int(len(sample)*0.7):].copy()

if SPLIT == "train":
    dataset = trainset
elif SPLIT == "valid":
    dataset = validset
elif SPLIT == "test":
    dataset = testset
else:
    raise ValueError(f"Wrong argument for split option {SPLIT}")

In [6]:
# Fill histograms
for idx in dataset.index:
    if dataset.loc[idx, 'label'] == 0:
        for feature in configs.keys(): bkgHists[feature].Fill(dataset.loc[idx, feature])
    else:
        for feature in configs.keys(): sigHists[feature].Fill(dataset.loc[idx, feature])

In [7]:
# save histograms to root file
outPath = f"{WORKDIR}/outputs/ex0/input_distribution_{SPLIT}.root"
os.makedirs(os.path.dirname(outPath), exist_ok=True)
out = ROOT.TFile(outPath, "recreate")
out.cd()
for hist in sigHists.values(): hist.Write()
for hist in bkgHists.values(): hist.Write()
out.Close()

In [14]:
# plot single feature
text = ROOT.TLatex()
def setCOMText():
    text.SetTextSize(0.035)
    text.SetTextFont(42)
    
def setCMSText():
    text.SetTextSize(0.04)
    text.SetTextFont(61)
    
def setWIPText():
    text.SetTextSize(0.036)
    text.SetTextFont(52)
    
def drawFeature(feature, x_title):
    # load histograms
    f = ROOT.TFile.Open(f"{WORKDIR}/outputs/ex0/input_distribution_{SPLIT}.root")
    sigHist = f.Get(f"sig_{feature}"); sigHist.SetDirectory(0)
    bkgHist = f.Get(f"bkg_{feature}"); bkgHist.SetDirectory(0)
    f.Close()

    # normalize to unit
    sigHist.Scale(1. / sigHist.Integral())
    bkgHist.Scale(1. / bkgHist.Integral())

    # histogram settings
    maximum = max(sigHist.GetMaximum(), bkgHist.GetMaximum())
    sigHist.SetStats(0)
    sigHist.SetLineColor(ROOT.kBlack); sigHist.SetLineWidth(3)
    sigHist.GetXaxis().SetTitle(x_title)
    sigHist.GetYaxis().SetRangeUser(0.0001, maximum*15)

    bkgHist.SetLineColor(ROOT.kBlue); bkgHist.SetLineWidth(3)

    legend = ROOT.TLegend(0.65, 0.65, 0.85, 0.8)
    legend.SetFillStyle(0)
    legend.SetBorderSize(0)
    legend.AddEntry(sigHist, "TT", "l")
    legend.AddEntry(bkgHist, "QCD", "l")
    
    # combine elements
    c = ROOT.TCanvas("canvas", "", 1600, 1500)
    c.cd()
    c.SetLogy()
    sigHist.Draw("hist")
    bkgHist.Draw("hist&same")
    legend.Draw()
    setCOMText(); text.DrawLatexNDC(0.75, 0.91, "(13.6 TeV)")
    setCMSText(); text.DrawLatexNDC(0.1, 0.91, "CMS")
    setWIPText(); text.DrawLatexNDC(0.2, 0.91, "Private Work")
    c.RedrawAxis()
    c.SaveAs(f"{WORKDIR}/outputs/ex0/plots/{feature}_logy.png")

In [15]:
drawFeature("j1_energy", "E(j1) [GeV]")
drawFeature("j1_px", "p_{x}(j1) [GeV]")
drawFeature("j1_py", "p_{y}(j1) [GeV]")
drawFeature("j1_pz", "p_{z}(j1) [GeV]")
drawFeature("j1_chEmEF", "charged EM energy fraction (j1)")
drawFeature("j1_chHEF", "charged Hadronic energy fraction (j1)")
drawFeature("j1_neEmEF", "neutral EM energy fraction (j1)")
drawFeature("j1_neHEF", "neutral Hadronic energy fraction (j1)")
drawFeature("j1_chEmEF", "charged EM energy fraction (j1)")
drawFeature("j1_muEF", "muon energy fraction (j1)")
drawFeature("j1_btagDeepFlavB", "DeepFlavB (j1)")
drawFeature("j1_btagDeepFlavQG", "DeepFlavQG (j1)")
drawFeature("j2_energy", "E(j2) [GeV]")
drawFeature("j2_px", "p_{x}(j2) [GeV]")
drawFeature("j2_py", "p_{y}(j2) [GeV]")
drawFeature("j2_pz", "p_{z}(j2) [GeV]")
drawFeature("j2_chEmEF", "charged EM energy fraction (j2)")
drawFeature("j2_chHEF", "charged Hadronic energy fraction (j2)")
drawFeature("j2_neEmEF", "neutral EM energy fraction (j2)")
drawFeature("j2_neHEF", "neutral Hadronic energy fraction (j2)")
drawFeature("j2_chEmEF", "charged EM energy fraction (j2)")
drawFeature("j2_muEF", "muon energy fraction (j2)")
drawFeature("j2_btagDeepFlavB", "DeepFlavB (j2)")
drawFeature("j2_btagDeepFlavQG", "DeepFlavQG (j2)")
drawFeature("j3_energy", "E(j3) [GeV]")
drawFeature("j3_px", "p_{x}(j3) [GeV]")
drawFeature("j3_py", "p_{y}(j3) [GeV]")
drawFeature("j3_pz", "p_{z}(j3) [GeV]")
drawFeature("j3_chEmEF", "charged EM energy fraction (j3)")
drawFeature("j3_chHEF", "charged Hadronic energy fraction (j3)")
drawFeature("j3_neEmEF", "neutral EM energy fraction (j3)")
drawFeature("j3_neHEF", "neutral Hadronic energy fraction (j3)")
drawFeature("j3_chEmEF", "charged EM energy fraction (j3)")
drawFeature("j3_muEF", "muon energy fraction (j3)")
drawFeature("j3_btagDeepFlavB", "DeepFlavB (j3)")
drawFeature("j3_btagDeepFlavQG", "DeepFlavQG (j3)")
drawFeature("j4_energy", "E(j4) [GeV]")
drawFeature("j4_px", "p_{x}(j4) [GeV]")
drawFeature("j4_py", "p_{y}(j4) [GeV]")
drawFeature("j4_pz", "p_{z}(j4) [GeV]")
drawFeature("j4_chEmEF", "charged EM energy fraction (j4)")
drawFeature("j4_chHEF", "charged Hadronic energy fraction (j4)")
drawFeature("j4_neEmEF", "neutral EM energy fraction (j4)")
drawFeature("j4_neHEF", "neutral Hadronic energy fraction (j4)")
drawFeature("j4_chEmEF", "charged EM energy fraction (j4)")
drawFeature("j4_muEF", "muon energy fraction (j4)")
drawFeature("j4_btagDeepFlavB", "DeepFlavB (j4)")
drawFeature("j4_btagDeepFlavQG", "DeepFlavQG (j4)")
drawFeature("avg_deltaR", "<#Delta R>")
drawFeature("Nj", "jet multiplicity")
drawFeature("HT", "HT [GeV]")

Info in <TCanvas::Print>: png file /home/choij/workspace/SNU-project/4.SNU-CMS/Tutorials/MachineLearning/outputs/ex0/plots/j1_energy_logy.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas
Info in <TCanvas::Print>: png file /home/choij/workspace/SNU-project/4.SNU-CMS/Tutorials/MachineLearning/outputs/ex0/plots/j1_px_logy.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas
Info in <TCanvas::Print>: png file /home/choij/workspace/SNU-project/4.SNU-CMS/Tutorials/MachineLearning/outputs/ex0/plots/j1_py_logy.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas
Info in <TCanvas::Print>: png file /home/choij/workspace/SNU-project/4.SNU-CMS/Tutorials/MachineLearning/outputs/ex0/plots/j1_pz_logy.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas
Info in <TCanvas::Print>: png file /home/choij/workspace/SNU-project/4.SNU-CMS/T